# Assignment 2 - Sequential Data Analysis
You are given `trace.log` a file of log traces generated by `bpftrace` using the `provenance.bt` script.
```bash
sudo bpftrace provenance.bt > trace.log
```

We will use these traces to generate a provenance graph, create queries on this provenance graph and reconstruct the attack that was captured using these log traces.

## Assignment submission
To submit this assignment, you will have to
- Upload the final version of your notebook (including the outputs) to Canvas.

## Flag
**Note:** Finding and/or submitting the flag is not a requirement for submitting the assignment.

During this assignment, you can find a base64-encoded flag. If you find this flag, you can upload it to  the https://security.eemcs.utwente.nl server to gain points. You will need to register an account if you haven't done so already. If you run into issues with registering an account, please contact the teacher of the course.

Navigate to `Challenges` -> `Cyber Data Analytics` -> `Assignment 2 - Sequential Data Analysis`. Here you can submit the flag you found.

**Note:** Finding and/or submitting the flag is not a requirement for submitting the assignment.

## Libraries
You will need the following Python libraries for this assignment:
- [networkx](https://networkx.org/)

In [1]:
import networkx as nx
import pandas as pd

## Constructing the provenance graph
The `trace.log` file traces 6 different system calls:
- [open()](https://www.man7.org/linux/man-pages/man2/openat.2.html), when opening a filepath, it returns the file descriptor.
  - `[<datetime>]::<program>[PID=<pid>]::open("<filepath>")-><file_descriptor>`
- [close()](https://www.man7.org/linux/man-pages/man2/close.2.html), when closing a filepath, it returns the file descriptor.
  - `[<datetime>]::<program>[PID=<pid>]::close(<file_descriptor>)`
- [read()](https://www.man7.org/linux/man-pages/man2/read.2.html), when reading from a file descriptor.
  - `[<datetime>]::<program>[PID=<pid>]::read(<file_descriptor>)`
- [write()](https://www.man7.org/linux/man-pages/man2/write.2.html), when writing to a file descriptor.
  - `[<datetime>]::<program>[PID=<pid>]::write(<file_descriptor>)`
- [fork()](https://www.man7.org/linux/man-pages/man2/fork.2.html), when a program forks (spawns) a child program.
  - `[<datetime>]::<program>[PID=<pid>]::fork(<child_pid>)`
- tcp(), a special log that occurs when a program uses a TCP connection.
  - `[<datetime>]::<program>[PID=<pid>]::tcp("<src>:<sport>", "<dst>:<dport>")`
 
The `<fields>` in the logs have the following format:
- `<datetime>`: `str` (`YYYY-MM-DD HH:MM:SS.μs`)
- `<program>`: `str`
- `<pid>`: `int`
- `<filepath>`: `str`
- `<file_descriptor>`: `int`
- `<child_pid>`: `int`
- `<src>`: `str`
- `<sport>`: `int`
- `<dst>`: `str`
- `<dport>`: `int`

**Question 1. [1.5pts]** Compile 6 regular expressions (1 for each system call) that parse the log files by capturing all `<fields>` according to the definitions given above.

In [2]:
import re


DATETIME = r"(?P<datetime>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+)"
PROGRAM  = r"(?P<program>[^\[]+)"
PID      = r"(?P<pid>\d+)"

PREFIX = rf"\[{DATETIME}\]::{PROGRAM}\[PID={PID}\]::"

OPEN_RE  = re.compile(rf'{PREFIX}open\("(?P<filepath>[^"]+)"\)->(?P<fd>-?\d+)')
CLOSE_RE = re.compile(rf'{PREFIX}close\((?P<fd>-?\d+)\)')
READ_RE  = re.compile(rf'{PREFIX}read\((?P<fd>-?\d+)\)')
WRITE_RE = re.compile(rf'{PREFIX}write\((?P<fd>-?\d+)\)')
FORK_RE  = re.compile(rf'{PREFIX}fork\((?P<child_pid>-?\d+)\)')
TCP_RE   = re.compile(
    rf'{PREFIX}tcp\("(?P<src>[^":]+):(?P<sport>\d+)",\s*'
    rf'"(?P<dst>[^":]+):(?P<dport>\d+)"\)'
)

SYSCALL_REGEXES = {
    "open":  OPEN_RE,
    "close": CLOSE_RE,
    "read":  READ_RE,
    "write": WRITE_RE,
    "fork":  FORK_RE,
    "tcp":   TCP_RE,
}


def parse_line(line: str):
    for name, regex in SYSCALL_REGEXES.items():
        m = regex.search(line)
        if m:
            return name, m.groupdict()
    return None, None

We can use the previous regular expressions to construct a provenance graph from the logs in `trace.log`. To this end, we are interested in the following nodes and edges:

**Nodes:**
- Process, this shows us which process initiates which system calls.
- Files, this shows us which files have been accessed.
- Network connections, this shows us which network connections have been made.

**Edges:**
- Read actions, showing us which processes read a file.
- Write actions, showing us which processes wrote to a file.
- Fork actions, showing us which process spawned a child process.
- TCP connections, showing us which process

**NB:** Both processes and files are identified by an integer, make sure you distinguish both in your provenance graph.
**NB:** Also note that if a file descriptor is closed, it can be reused as a descriptor for a *different* file.

**Question 2. [2.5pts]** Use the regular expressions to construct a provenance graph from the logs in `trace.log`.

In [3]:
graph = nx.DiGraph()

In [4]:
fd_table = {}

def add_file_access(graph, action, info):
    pid, fd = info["pid"], info["fd"]
    if action == "open":
        node = info["filepath"]
        if node not in graph.nodes:
            graph.add_node(node, type="file", filepath=info["filepath"])
        fd_table[(pid, fd)] = node
        graph.add_edge(pid, node, action="open")
    elif action == "close":
        node = fd_table.pop((pid, fd), None)
        if node is not None:
            graph[pid][node]['action'] += " | close"
    else:  # read / write
        node = fd_table.get((pid, fd))
        if node is not None:
            graph[pid][node]['action'] += f" | {action}"


In [5]:
def add_syscall(graph, action, info):
    if info["pid"] not in graph.nodes:
        graph.add_node(info["pid"], program=info["program"], type="process")
    elif info["pid"] in graph.nodes and graph.nodes[info["pid"]].get("program") is None:
        graph.nodes[info["pid"]]["program"] = info["program"]
        
    if action == "fork":
        if info["child_pid"] not in graph.nodes:
            graph.add_node(info["child_pid"], type="process")
        graph.add_edge(info["pid"], info["child_pid"], action=action)
    elif action == "tcp":
        if info["dst"] not in graph.nodes:
            graph.add_node(info["dst"], type="ip", dst_port=info["dport"], src_port=info['sport'], src_ip=info['src'])
        graph.add_edge(info["pid"], info["dst"], action=action)
    else:
        add_file_access(graph, action, info)


In [6]:
with open('./trace.log') as f:
    for line in f:
        action, info = parse_line(line)
        if action is not None:    
            add_syscall(graph, action, info)
        else:
            print(f"Unrecognized line: {line.strip()}")


In [7]:
graph.nodes(data=True)

NodeDataView({'53499': {'program': 'dockerd', 'type': 'process'}, '402552': {'program': 'dockerd', 'type': 'process'}, '2475683': {'program': 'containerd', 'type': 'process'}, '2475721': {'program': 'multipathd', 'type': 'process'}, '2475682': {'program': 'irqbalance', 'type': 'process'}, '/proc/interrupts': {'type': 'file', 'filepath': '/proc/interrupts'}, '/proc/stat': {'type': 'file', 'filepath': '/proc/stat'}, '/proc/irq/33/smp_affinity': {'type': 'file', 'filepath': '/proc/irq/33/smp_affinity'}, '/proc/irq/16/smp_affinity': {'type': 'file', 'filepath': '/proc/irq/16/smp_affinity'}, '/proc/irq/35/smp_affinity': {'type': 'file', 'filepath': '/proc/irq/35/smp_affinity'}, '/proc/irq/36/smp_affinity': {'type': 'file', 'filepath': '/proc/irq/36/smp_affinity'}, '/proc/irq/37/smp_affinity': {'type': 'file', 'filepath': '/proc/irq/37/smp_affinity'}, '/proc/irq/25/smp_affinity': {'type': 'file', 'filepath': '/proc/irq/25/smp_affinity'}, '/proc/irq/31/smp_affinity': {'type': 'file', 'filepat

## Analysis
Now that we have constructed the graph, we will investigate what attack was executed.

**Question 3. [0.5pts]** How many different files were accessed in the provenance graph?

In [8]:
unique_files = [data for node, data in graph.nodes(data=True) if data.get('type') == 'file']
len(unique_files)

1011

We received a complaint that alice cannot access here records stored in `alice.txt`.

**Question 4a. [1pts]** Write a function to find all processes that were both directly or indirectly responsible for accessing a file. Use this function to find all processes that were both directly or indirectly responsible for accessing `finance/alice.txt`.

In [9]:
def find_process_chain_file_access(provenance: nx.DiGraph, filename: str) -> list:
    try: 
        file_node = [node for node in provenance.nodes() if filename in node][0]
    except IndexError:
        return "File not found in provenance graph."
    process_chain = []
    for pred in nx.ancestors(provenance, file_node):
        if provenance.nodes[pred].get('type') == 'process':
            node = provenance.nodes[pred]
            edge = provenance.get_edge_data(pred, file_node)
            action = edge.get('action') if edge else None
            process_chain.append({
                'pid': pred,
                'program': node.get('program'),
                'action': action,
            })
    return process_chain

print(find_process_chain_file_access(graph, 'finance/alice.txt'))

[{'pid': '2477912', 'program': 'gunicorn', 'action': None}, {'pid': '2477950', 'program': 'python3', 'action': None}, {'pid': '2477967', 'program': 'python3', 'action': 'open | read | read | close'}, {'pid': '2477949', 'program': 'gunicorn', 'action': None}]


**Question 4b. [0.5pts]** What process was used as the initial access point of the attacker?

**Question 5. [1pts]** Write a function that, given a pid of the initial access point, finds all files that were accessed (both directly or indirectly) when using this initial access point. What other files were accessed (both directly or indirectly) from the initial access point found in question 4b?

In [10]:
def files_accessed_from_initial_access(provenance: nx.DiGraph, pid: int) -> list:
    node = provenance.nodes[str(pid)]
    process_chain = []
    for next_node in nx.descendants(provenance, str(pid)):
        if provenance.nodes[next_node].get('type') == 'file':
            node = provenance.nodes[next_node]
            edge = provenance.get_edge_data(str(pid), next_node)
            action = edge.get('action') if edge else None
            open = 0
            read = 0
            write = 0
            close = 0
            if action and 'open' in action:
                open = 1
            if action and 'read' in action:
                read = 1
            if action and 'write' in action:
                write = 1
            if action and 'close' in action:
                close = 1
            process_chain.append({
                'pid': str(pid),
                'program': node.get('filepath'),
                'action': action,
                'open': open,
                'read': read,
                'write': write,
                'close': close
            })
    df = pd.DataFrame(process_chain)
    return df

files_accessed_from_initial_access(graph, 2477967)

,pid,program,action,open,read,write,close
0,2477967,/usr/lib/python3.10/__pycache__/codecs.cpython...,open | read | read | close,1,1,0,1
1,2477967,/usr/lib/python3.10/__pycache__/lzma.cpython-3...,open | read | read | close,1,1,0,1
2,2477967,/usr/lib/python3.10/urllib,open | close,1,0,0,1
3,2477967,/lib/x86_64-linux-gnu/libm.so.6,open | read | close,1,1,0,1
4,2477967,/usr/lib/python3.10/__pycache__/sre_parse.cpyt...,open | read | read | close,1,1,0,1
...,...,...,...,...,...,...,...
82,2477967,/usr/lib/python3.10/lib-dynload/_bz2.cpython-3...,open | read | close,1,1,0,1
83,2477967,/usr/lib/python3.10/__pycache__/shutil.cpython...,open | read | read | close,1,1,0,1
84,2477967,finance/bob.txt,open | read | read | close,1,1,0,1
85,2477967,/usr/local/lib/python3.10/dist-packages,open | close,1,0,0,1


**Question 6. [1pts]** It looks like the `finance/` directory is holds interesting information. Which python program directly read and/or wrote these files?

**Question 7. [0.5pts]** From which IP address was the python script from question 6 downloaded?

*Hint: Files can be accessed through full or relative paths.*

In [11]:
find_process_chain_file_access(graph, 'encryptor.py')

[{'pid': '2477949', 'program': 'gunicorn', 'action': None},
 {'pid': '2477912', 'program': 'gunicorn', 'action': None},
 {'pid': '2477950', 'program': 'python3', 'action': None},
 {'pid': '2477961', 'program': 'scp', 'action': 'open | write | close'}]

In [12]:
parents = nx.ancestors(graph, '2477961')
for parent in parents:
    print(graph.nodes[parent])

{'program': 'gunicorn', 'type': 'process'}
{'type': 'process', 'program': 'python3'}
{'type': 'process', 'program': 'gunicorn'}


In [13]:
children = nx.descendants(graph, '2477961')
for child in children:
    if graph.nodes[child].get('type') == 'ip':
        print(child, graph.nodes[child])

130.89.6.147 {'type': 'ip', 'dst_port': '8000', 'src_port': '39126', 'src_ip': '130.89.6.92'}


**Question 8. [0.5pts]** Describe the attack in terms of [MITRE ATT&CK](https://attack.mitre.org/) tactics and techniques that were used.

## Automation
We have performed a manual analysis for the given attack, however, ideally we would like to have an automated detection of specific attack patterns.

**Question 9. [1pts]** Given a provenance graph, write a function that automatically detects which processes (nodes in graph) are directly executing a ransomware attack. You can assume a ransomware attack reads and writes at least 5 files.

In [21]:
def detect_ransomware(provenance: nx.DiGraph):
    for node, data in provenance.nodes(data=True):
        files_accessed = files_accessed_from_initial_access(provenance, node)
        if files_accessed.empty:
            continue
        write_actions = files_accessed['write'].sum()
        read_actions = files_accessed['read'].sum()
        if write_actions >= 5 and read_actions >= 5:
            print(f"Process {node} ({data.get('program')}) conducted {write_actions} write actions and {read_actions} read actions, which indicated a ransomware attack.")

In [22]:
detect_ransomware(graph)

Process 2477967 (python3) conducted 6 write actions and 63 read actions, which indicated a ransomware attack.


Base64 Flag

In [23]:
import base64

def try_decode_base64(string):
    try:
        string_bytes = base64.b64decode(string)
        decoded_string = string_bytes.decode("ascii")
        return decoded_string
    except:
        return None

In [ ]:
for node in graph.nodes():
    try_dec = try_decode_base64(node)
    if try_dec:
        print(f"Decoded string: {try_dec}")
    split = node.split('/')
    for part in split:
        try_dec_part = try_decode_base64(part)
        if try_dec_part:
            print(f"Decoded part: {try_dec_part}")
            print(f"node is {node}")


Decoded part: ns0mw4
node is /home/victim/cda/server/records/bnMwbXc0
Decoded part: r3}
node is /home/victim/cda/server/records/cjN9
Decoded part: s1c_r4
node is /home/victim/cda/server/records/czFjX3I0
Decoded part: z{k
node is /usr/local/lib/python3.10/dist-packages/idna-3.4.dist-info/entr
Decoded part: {
node is /usr/local/lib/python3.10/dist-packages/gitdb/__pycache__/exc.c
Decoded part: THS{b4
node is /home/victim/cda/server/records/VEhTe2I0


Submitted the flag